In [4]:
from wrappers import ModelWrapper
from model import ResMultimodalModel
import torch
import numpy as np
import matplotlib.pyplot as plt
from data import COVID19ARPreRes, ISUPPreRes, DVMPreRes
from torchvision import transforms as T
from torch.utils.data import DataLoader
import seaborn as sns
from tqdm import tqdm

In [2]:
# global explanation
# source_idx = [0,3,4,5,6,7,8,9,10,11,12,13,14,15]
# target_idx = [0,1,1,1,2,3,3,4,4,5,6,6,6,0]
source_idx = [0,3,4,5,6,7,8,9,10,11,12,13,14,15]
target_idx = [0,3,4,5,6,7,8,9,10,11,12,13,14,15]
model = ResMultimodalModel(tab_emb_dim=1024, 
                           num_classes=2, 
                           feat_dim = 1024, 
                           fusion='attn', 
                         image_encoder='rn50_clip_res', 
                           tab_encoder='identical', 
                           frozen_tab=True, 
                         complex_classifier = False, 
                           nhead=1, 
                           zero_conv = True,
                         source_idx = source_idx, 
                         target_idx = target_idx,
                         first_layer_finetune = True
                          )

In [3]:
wrapper_config = {
    'wrapper_input': 'dict',
    'model_input': 'dict',
    'model_output': 'tensor',
    'kwd': None
}
model = ModelWrapper(model, wrapper_config = wrapper_config, device='cuda')

In [4]:
tfs = {
    'img_tf': T.Compose([T.Resize((224, 224)), 
                         T.ToTensor(),                 
                         T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
                        ]), 
    'tab_tf': lambda x, y: y
}

In [5]:
# dataset = COVID19ARPreRes(split = 'test', transforms=tfs, 
#                           kwd='clip_gpt', 
#                           root_dir='data/covid19_ar', 
#                           modal=['img', 'tab'], 
#                           preload_images=False, 
#                           sample=False,)
dataset = COVID19ARPreRes(split = 'test', transforms=tfs, 
                          kwd='clip_cell', 
                          root_dir='data/covid19_ar', 
                          modal=['img', 'tab'], 
                          preload_images=False, 
                          sample=False,)

In [6]:
loader = DataLoader(dataset, shuffle=False, num_workers=8, batch_size=32)

In [7]:
model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/ava2/models/best_model_1700107478_163354_epoch7.t7')['state_dict'])
# model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/ava3/models/last_model_1700109610_2331324_epoch14.t7')['state_dict'])

<All keys matched successfully>

In [8]:
# local explantions
logits = []
for d in tqdm(loader):
    with torch.no_grad():
        model.eval()
        logit = model(d)
    logits.append(logit['logit'].detach().cpu().numpy())

100%|█████████████████████████████████████████| 215/215 [00:26<00:00,  8.05it/s]


In [ ]:
corr_attn = np.concatenate(model.model.attentive_pool.corr_attn, axis=0)
con_attn = np.concatenate(model.model.attentive_pool.con_attn, axis=0)

In [9]:
feat = np.concatenate(model.model.attentive_pool.tab_emb, axis=0)
np.save('covid_tab_emb.npy', feat)

In [ ]:
np.save('covid_corr_attn.npy', corr_attn)
np.save('covid_con_attn.npy', con_attn)

In [ ]:
np.mean(corr_attn[:,:,0], axis=0)/16

In [ ]:
con_attn.mean(0)[...,0]

In [ ]:
plt.figure()
sns.barplot(x = list(range(7)), y = np.mean(corr_attn[:,:,0], axis=0)/7, saturation=0.5, color='salmon')
plt.show()

In [ ]:
plt.figure()
def sigmoid(x):                                        
    return 1 / (1 + np.exp(-x))
sns.barplot(x = list(range(7)), y = sigmoid(np.mean(con_attn[:,:,0], axis=0)), saturation=0.5, color='salmon')
plt.show()

In [5]:
# global explanation
source_idx=[0,1,2,3,4,5,6,7,8,9,10,11,12]
target_idx=[1,1,1,1,2,2,2,3,3,4,4,5,6]
model = ResMultimodalModel(tab_emb_dim=1024, 
                           num_classes=286, 
                           feat_dim = 1024, 
                           fusion='attn', 
                         image_encoder='rn50_clip_res', 
                           tab_encoder='identical', 
                           frozen_tab=True, 
                         complex_classifier = False, 
                           nhead=1, 
                           zero_conv = True,
                         source_idx = source_idx, 
                         target_idx = target_idx,
                         first_layer_finetune = True
                          )

In [6]:
wrapper_config = {
    'wrapper_input': 'dict',
    'model_input': 'dict',
    'model_output': 'tensor',
    'kwd': None
}
model = ModelWrapper(model, wrapper_config = wrapper_config, device='cuda')
tfs = {
    'img_tf': T.Compose([T.Resize((224, 224)), 
                         T.ToTensor(),                 
                         T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
                        ]), 
    'tab_tf': lambda x, y: y
}

In [7]:
dataset = DVMPreRes(split = 'test', transforms=tfs, 
                          kwd='clip_gpt', 
                          root_dir='data/dvm_car', 
                          modal=['img', 'tab'], 
                          preload_images=False, 
                          sample=False,)

In [8]:
loader = DataLoader(dataset, shuffle=False, num_workers=8, batch_size=32)

In [9]:
model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/use1/models/best_model_1699964607_0250363_epoch56.t7')['state_dict'])

<All keys matched successfully>

In [ ]:
# local explantions
for d in tqdm(loader):
    with torch.no_grad():
        model.eval()
        model(d)

 94%|████████████████████████████████████▍  | 2580/2757 [04:02<00:16, 10.56it/s]

In [ ]:
feat = np.concatenate(model.model.attentive_pool.tab_emb, axis=0)
np.save('dvm_tab_emb.npy', feat)

In [ ]:
corr_attn = np.concatenate(model.model.attentive_pool.corr_attn, axis=0)
con_attn = np.concatenate(model.model.attentive_pool.con_attn, axis=0)
np.save('dvm_corr_attn.npy', corr_attn)
np.save('dvm_con_attn.npy', con_attn)

In [13]:
corr_attn.shape

(88207, 7, 1)

In [14]:
print(np.mean(corr_attn[:,:,0], axis=0)/7)
print(con_attn.mean(0)[...,0])

[0.16145669 0.1894931  0.0752411  0.1003601  0.1870724  0.0451234
 0.24125625]
[7.1830589e-01 8.9028490e-01 7.7895463e-01 5.1365873e-06 4.1102460e-01
 9.9741334e-01 9.6336466e-01]


In [12]:
# global explanation
source_idx = [3,4,5,6,7,8,9,10,11,12,13]
target_idx = [3,3,4,4,4,5,5,6,6,7,7]
model = ResMultimodalModel(tab_emb_dim=1024, 
                           num_classes=3, 
                           feat_dim = 1024, 
                           fusion='attn', 
                         image_encoder='rn50_clip_res', 
                           tab_encoder='identical', 
                           frozen_tab=True, 
                         complex_classifier = False, 
                           nhead=4, 
                           zero_conv = True,
                         source_idx = source_idx, 
                         target_idx = target_idx,
                         first_layer_finetune = True
                          )

In [13]:
wrapper_config = {
    'wrapper_input': 'dict',
    'model_input': 'dict',
    'model_output': 'tensor',
    'kwd': None
}
model = ModelWrapper(model, wrapper_config = wrapper_config, device='cuda')

In [14]:
tfs = {
    'img_tf': T.Compose([T.Resize((224, 224)), 
                         T.ToTensor(),                 
                         T.Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
                        ]), 
    'tab_tf': lambda x, y: y
}
dataset = ISUPPreRes(split = 'test', transforms=tfs, 
                          kwd='clip_gpt', 
                          modal=['img', 'tab'], 
                          preload_images=False, 
                          sample=False,)

In [15]:
loader = DataLoader(dataset, shuffle=False, num_workers=8, batch_size=32)

In [16]:
model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/avb1/models/best_model_1700039670_611354_epoch14.t7')['state_dict'])
# model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/america/models/best_model_1700191037_0703747_epoch11.t7')['state_dict'])

<All keys matched successfully>

In [17]:
# local explantions
logits = []
for d in tqdm(loader):
    with torch.no_grad():
        model.eval()
        logit = model(d)
    logits.append(logit['logit'].detach().cpu().numpy())

100%|█████████████████████████████████████████████| 6/6 [00:14<00:00,  2.46s/it]


In [18]:
feat = np.concatenate(model.model.attentive_pool.tab_emb, axis=0)
np.save('isup_tab_emb.npy', feat)

In [ ]:
corr_attn = np.concatenate(model.model.attentive_pool.corr_attn, axis=0)
con_attn = np.concatenate(model.model.attentive_pool.con_attn, axis=0)

In [ ]:
np.save('isup_corr_attn.npy', corr_attn)
np.save('isup_con_attn.npy', con_attn)

In [ ]:
a = np.load('isup_corr_attn.npy')
np.mean(a[:,:,0], axis=0)/8

In [ ]:
a = np.load('isup_con_attn.npy')
plt.figure(figsize=(8, 3))
def sigmoid(x):                                        
    return 1 / (1 + np.exp(-x))
k = sigmoid(np.mean(a[:,:,0], axis=0))
k

In [ ]:
plt.figure(figsize=(8, 3))
k = np.mean(corr_attn[:,:,0], axis=0)/8
sns.barplot(x = list(range(8)), y = k, saturation=0.3, color='blue')
sns.barplot(x = list(range(8)), y = [0,0,0,0,k[4],0,0,0], saturation=0.5, color='salmon')
plt.xlabel('Sentence index')
plt.ylabel('Attention score')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 3))
def sigmoid(x):                                        
    return 1 / (1 + np.exp(-x))
k = sigmoid(np.mean(con_attn[:,:,0], axis=0))
sns.barplot(x = list(range(8)), y = k, saturation=0.3, color='blue')
sns.barplot(x = list(range(8)), y = [0,0,0,0,0,k[5],0,0], saturation=0.5, color='salmon')
plt.xlabel('Concept index')
plt.ylabel('Importance score')
plt.tight_layout()
plt.show()

In [ ]:
dataset = ISUPPreRes(split = 'test', transforms=tfs, 
                          kwd='clip_gpt', 
                          modal=['img', 'tab'], 
                          preload_images=False, 
                          sample=False,)
d = iter(loader).__next__()
image = d['image']
res_tabline = d['res_tabline']
main_tabline = d['main_tabline']
image.requires_grad = True
res_tabline.requires_grad = True
model.load_state_dict(torch.load('/home/lmx/Image-Tabular-Fusion/logs/america/models/best_model_1700191037_0703747_epoch11.t7')['state_dict'])
model.eval()
logit = model.model(main_tabline.cuda(), res_tabline.cuda(), image.cuda())

In [ ]:
tab_emb = model.model.attentive_pool.tab_emb

In [ ]:
tab_emb[0,4,:].sum().backward()

In [ ]:
label = d['label'].cuda()
loss = torch.nn.CrossEntropyLoss()(logit, label)

In [ ]:
loss.backward()

In [ ]:
image.grad.shape

In [ ]:
img = image[0,...].detach().cpu().numpy()
img = (img / 2) + 1

In [ ]:
grad = image.grad[0,...]
grad = torch.abs(grad).detach().cpu().numpy()
grad = (grad - grad.min()) / (grad.max() - grad.min())
plt.figure(figsize=(10, 5))
plt.subplot(1,3,1)
plt.imshow(grad[0,...], cmap='hot')
plt.axis('off')
plt.title('')
plt.subplot(1,3,2)
plt.imshow(img[0,...])
# plt.imshow(grad[1,...]*img[1,...], cmap='hot')
plt.axis('off')
plt.title('')
# plt.subplot(1,3,3)
# plt.imshow(grad[2,...]*img[2,...], cmap='hot')
# plt.axis('off')
# plt.title('')
plt.show()